In [17]:
%load_ext autoreload
%autoreload 2

import sys
import time
import joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import f1_score

# NEW in Phase 3 — imblearn Pipeline + samplers
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTETomek

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

RANDOM_STATE = 25
DATA_DIR    = Path.cwd().parent / 'data' / 'processed'
RESULTS_DIR = Path.cwd().parent / 'outputs' / 'results'
MODELS_DIR  = Path.cwd().parent / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
X_train = pd.read_parquet(DATA_DIR / 'X_train.parquet')
y_train = pd.read_parquet(DATA_DIR / 'y_train.parquet')['Victims_Condition']

# XGBoost needs integer labels
y_train_int, class_labels = pd.factorize(y_train, sort=True)
y_train_int = pd.Series(y_train_int, index=y_train.index)

print(f"X_train: {X_train.shape}")
print(f"Class labels: {list(class_labels)}")

X_train: (370466, 21)
Class labels: ['With dead victims', 'With injured victims', 'Without victims']


In [19]:
numeric_cols     = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_cols = X_train.select_dtypes(include=['category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=True), categorical_cols),
])

In [ ]:
def make_dt(class_weight=None):
    return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight=class_weight)

def make_cat(class_weight=None):
    return CatBoostClassifier(
        random_state=RANDOM_STATE, verbose=0,
        auto_class_weights='Balanced' if class_weight else None,
    )

def make_xgb():
    return XGBClassifier(
        tree_method='hist', n_jobs=-1,
        random_state=RANDOM_STATE, verbosity=0,
    )

# 6 samplers — the 7th "method" is class weighting, handled inside the classifier
samplers = {
    'none':        None,
    'RandomOver':  RandomOverSampler(random_state=RANDOM_STATE),
    'RandomUnder': RandomUnderSampler(random_state=RANDOM_STATE),
    'SMOTE':       SMOTE(random_state=RANDOM_STATE),
    'ADASYN':      ADASYN(random_state=RANDOM_STATE),
    'SMOTE+Tomek': SMOTETomek(random_state=RANDOM_STATE),
}

# Build 20 experiments: 3 classifiers × 6 samplers + 2 native class-weighted classifiers

experiments = []
for clf_name, make_clf in [('DecisionTree', make_dt), ('CatBoost', make_cat), ('XGBoost', make_xgb)]:
    for sampler_name, sampler in samplers.items():
        experiments.append((clf_name, sampler_name, make_clf(), sampler))

experiments.append(('DecisionTree', 'ClassWeight', make_dt(class_weight='balanced'), None))
experiments.append(('CatBoost',     'ClassWeight', make_cat(class_weight='balanced'), None))

print(f"Total pipeline-friendly experiments: {len(experiments)}")
print(f"Plus XGBoost+ClassWeight in Cell 5b => 21 total")

Total pipeline-friendly experiments: 20
Plus XGBoost+ClassWeight in Cell 5b => 21 total


In [21]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = []
total_start = time.time()

for i, (clf_name, method_name, clf, sampler) in enumerate(experiments, start=1):
    print(f"[{i:2d}/{len(experiments)}] {clf_name} + {method_name} ... ", end='', flush=True)
    t0 = time.time()

    # imblearn Pipeline runs the sampler INSIDE each CV fold's training data
    steps = [('pre', preprocessor)]
    if sampler is not None:
        steps.append(('sampler', sampler))
    steps.append(('clf', clf))
    pipe = ImbPipeline(steps)

    y_for_cv = y_train_int if clf_name == 'XGBoost' else y_train
    scores = cross_val_score(pipe, X_train, y_for_cv, cv=cv, scoring='f1_macro', n_jobs=-1)
    elapsed = time.time() - t0

    results.append({
        'classifier':      clf_name,
        'method':          method_name,
        'cv_macro_f1':     scores.mean(),
        'cv_macro_f1_std': scores.std(),
        'time_sec':        round(elapsed, 1),
    })
    print(f"macro-F1 = {scores.mean():.4f}  |  {elapsed:.1f}s")

print(f"\nCell 5 elapsed: {(time.time()-total_start)/60:.1f} min")

[ 1/20] DecisionTree + none ... macro-F1 = 0.4358  |  212.2s
[ 2/20] DecisionTree + RandomOver ... macro-F1 = 0.4316  |  460.4s
[ 3/20] DecisionTree + RandomUnder ... macro-F1 = 0.3838  |  17.7s
[ 4/20] DecisionTree + SMOTE ... macro-F1 = 0.4339  |  1112.6s
[ 5/20] DecisionTree + ADASYN ... macro-F1 = 0.4353  |  2615.5s
[ 6/20] DecisionTree + SMOTE+Tomek ... macro-F1 = 0.4354  |  32051.2s
[ 7/20] CatBoost + none ... macro-F1 = 0.4184  |  96.7s
[ 8/20] CatBoost + RandomOver ... macro-F1 = 0.4825  |  205.5s
[ 9/20] CatBoost + RandomUnder ... macro-F1 = 0.4697  |  26.5s
[10/20] CatBoost + SMOTE ... macro-F1 = 0.4588  |  933.0s
[11/20] CatBoost + ADASYN ... macro-F1 = 0.4521  |  2247.4s
[12/20] CatBoost + SMOTE+Tomek ... macro-F1 = 0.4581  |  31492.2s
[13/20] XGBoost + none ... macro-F1 = 0.4156  |  20.4s
[14/20] XGBoost + RandomOver ... macro-F1 = 0.4811  |  25.4s
[15/20] XGBoost + RandomUnder ... macro-F1 = 0.4687  |  7.3s
[16/20] XGBoost + SMOTE ... macro-F1 = 0.4310  |  208.4s
[17/20] 

In [22]:
# XGBoost multiclass doesn't accept a class_weight parameter directly.
# Class-weighting is done by computing per-sample weights from the training
# fold's class distribution and passing them via sample_weight at fit time.
# cross_val_score can't route fit_params to a pipeline step cleanly, so we
# write a small manual CV loop just for this one experiment.

print("[21/21] XGBoost + ClassWeight (manual CV) ... ", end='', flush=True)
t0 = time.time()

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
fold_scores = []

for tr_idx, va_idx in cv.split(X_train, y_train_int):
    X_tr, X_va = X_train.iloc[tr_idx], X_train.iloc[va_idx]
    y_tr, y_va = y_train_int.iloc[tr_idx], y_train_int.iloc[va_idx]

    pipe = ImbPipeline([('pre', preprocessor), ('clf', make_xgb())])
    sw = compute_sample_weight('balanced', y_tr)
    pipe.fit(X_tr, y_tr, clf__sample_weight=sw)
    fold_scores.append(f1_score(y_va, pipe.predict(X_va), average='macro', zero_division=0))

elapsed = time.time() - t0
results.append({
    'classifier':      'XGBoost',
    'method':          'ClassWeight',
    'cv_macro_f1':     np.mean(fold_scores),
    'cv_macro_f1_std': np.std(fold_scores),
    'time_sec':        round(elapsed, 1),
})
print(f"macro-F1 = {np.mean(fold_scores):.4f}  |  {elapsed:.1f}s")

[21/21] XGBoost + ClassWeight (manual CV) ... macro-F1 = 0.4804  |  19.0s


In [23]:
results_df = (pd.DataFrame(results)
              .sort_values('cv_macro_f1', ascending=False)
              .reset_index(drop=True))

results_df.to_csv(RESULTS_DIR / 'phase3_imbalance_experiments.csv', index=False)
print(results_df.head(10))

# Refit the top 3 combos on the full training set and save
top_3 = results_df.head(3)
print(f"\nTop 3:\n{top_3[['classifier', 'method', 'cv_macro_f1']].to_string(index=False)}")

for _, row in top_3.iterrows():
    clf_name, method_name = row['classifier'], row['method']
    tag = f"{clf_name}_{method_name}"

    if clf_name == 'XGBoost' and method_name == 'ClassWeight':
        # Same manual pattern as Cell 5b
        pipe = ImbPipeline([('pre', preprocessor), ('clf', make_xgb())])
        sw = compute_sample_weight('balanced', y_train_int)
        pipe.fit(X_train, y_train_int, clf__sample_weight=sw)
    else:
        # Find this combo in the experiments list and rebuild the pipeline
        match = next(e for e in experiments if e[0] == clf_name and e[1] == method_name)
        _, _, clf, sampler = match
        steps = [('pre', preprocessor)]
        if sampler is not None:
            steps.append(('sampler', sampler))
        steps.append(('clf', clf))
        pipe = ImbPipeline(steps)
        y_for_fit = y_train_int if clf_name == 'XGBoost' else y_train
        pipe.fit(X_train, y_for_fit)

    joblib.dump(pipe, MODELS_DIR / f'phase3_{tag}.joblib')
    print(f"  saved: models/phase3_{tag}.joblib")

top_3[['classifier', 'method', 'cv_macro_f1']].to_csv(
    RESULTS_DIR / 'phase3_top3.csv', index=False
)

     classifier       method  cv_macro_f1  cv_macro_f1_std  time_sec
0      CatBoost   RandomOver     0.482515         0.002138     205.5
1       XGBoost   RandomOver     0.481108         0.002236      25.4
2       XGBoost  ClassWeight     0.480378         0.001877      19.0
3      CatBoost  ClassWeight     0.479878         0.002056     101.5
4      CatBoost  RandomUnder     0.469702         0.002631      26.5
5       XGBoost  RandomUnder     0.468697         0.002877       7.3
6      CatBoost        SMOTE     0.458841         0.002674     933.0
7      CatBoost  SMOTE+Tomek     0.458072         0.003201   31492.2
8      CatBoost       ADASYN     0.452112         0.002935    2247.4
9  DecisionTree         none     0.435800         0.001746     212.2

Top 3:
classifier      method  cv_macro_f1
  CatBoost  RandomOver     0.482515
   XGBoost  RandomOver     0.481108
   XGBoost ClassWeight     0.480378
  saved: models/phase3_CatBoost_RandomOver.joblib
  saved: models/phase3_XGBoost_RandomOv